In [ ]:
import torch
import numpy as np
from mhnlib.fixed_points import get_symmetric_stability_matrix_gram, get_entropies
from mhnlib.dynamics import DualDeterministicDynamics
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from math import log2, log, sqrt

In [ ]:
%load_ext autoreload

# Uncorrelated patterns

## Fixed N, K runs

### Create patterns

In [ ]:
%autoreload 2
N = 32
log2_N = log2(N)
Ks = [int(2**k) for k in range(int(log2_N)-2, int(log2_N)+5)]

In [ ]:
#betas_ic = torch.logspace(-10, 0, steps=200,)
#epsilons_ic = torch.logspace(-5, 0, steps=51,)
#uniform_weights = torch.ones(K)/K
#for patterns_idx, patterns in tqdm(enumerate(patterns_collection)):
#    dyn_patterns =  DualDeterministicDynamics(patterns, torch.zeros(K), requires_grad=False)
#    stability_matrix, proj = get_symmetric_stability_matrix_gram(gram, uniform_weights.to(gram.device), return_proj=True)
#    sm_vals, sm_vecs = torch.linalg.eigh(stability_matrix)
#    #proj_sm_vecs = proj.T @ sm_vecs
#    proj_sm_vecs = sm_vecs - proj_sm_vecs.mean(dim=0)
#    directions = proj_sm_vecs / proj_sm_vecs.norm(dim=0)
#    w0 = uniform_weights[None,:] + epsilons_ic[:, None] * directions[None, :,-1]
#    w0 = torch.clamp(w0, min=0.0)
#    w0 = w0 / w0.sum(dim=1, keepdim=True)
#    w_ic = dyn_patterns.fixed_points_quench(w0, betas_ic, num_iterations=10000)
#    break

In [ ]:
betas = torch.logspace(-2,2, steps=201)
pbar = tqdm(Ks)
device = "cuda" if torch.cuda.is_available() else "cpu"
#if torch.backends.mps.is_available():
#    device = "mps"
for K in pbar:
    torch.manual_seed(1101252)
    N_samples = max(2*K, 128)
    patterns_collection = torch.randn(N_samples, K, N)/torch.tensor(N).sqrt()
    uniform_weights = torch.ones(K)/K
    weights_fp = []
    weights_fp_centered = []
    for patterns_idx, patterns in enumerate(patterns_collection):
        pbar.set_description(f"K={K}, sample={patterns_idx+1}/{N_samples}")
        dyn_patterns =  DualDeterministicDynamics(patterns.to(device), biases=None, requires_grad=False).to(device)
        w_fp = dyn_patterns.integrate(uniform_weights[None,:].to(device), betas.to(device), num_iterations=3000).to('cpu')
        dyn_patterns_centered = DualDeterministicDynamics((patterns - patterns.mean(dim=0)).to(device), biases=None, requires_grad=False).to(device)
        w_fp_centered = dyn_patterns_centered.integrate(uniform_weights[None,:].to(device), betas.to(device), num_iterations=3000).to('cpu')
        weights_fp.append(w_fp)
        weights_fp_centered.append(w_fp_centered)
    weights_fp = torch.cat(weights_fp, dim=0)
    weights_fp_centered = torch.cat(weights_fp_centered, dim=0)
    H_fp = get_entropies(weights_fp).mean(dim=0)
    H_fp_centered = get_entropies(weights_fp_centered).mean(dim=0)
    torch.save({ "N" : N, "K": K, 
                "betas": betas.cpu().numpy(),
                 "patterns": patterns_collection.cpu().numpy(), 
                 "weights_fp": weights_fp.cpu().numpy(), 
                 "weights_fp_centered": weights_fp_centered.cpu().numpy()}, 
                 "local_data/random_patterns_N={}_K={}.pt".format(N, K))
    #fig, ax = plt.subplots()
    #ax.plot(betas.cpu(), H_fp.cpu(), label="FP")
    #ax.plot(betas.cpu(), H_fp_centered.cpu(), label="FP centered")
    #ax.set_xscale("log")
    #ax.set_xlabel("beta")
    #ax.set_ylabel("entropy")
    #ax.set_title(f"K={K}")
    #ax.legend()
    #plt.show()

# 1-block hierarchical

## Fixed N, K runs

In [ ]:
%autoreload 2
N = 32
log2_N = log2(N)
Ks = [int(2**k) for k in range(int(log2_N)-2, int(log2_N)+5)]
Ms = { K : list(np.unique([int(2**k) for k in range(int(log2(K))-5, int(log2(K))+1)]))  for K in Ks }
rho0 = 0.1
rho1 = 0.9

In [ ]:
betas = torch.logspace(-2,2, steps=201)
pbar = tqdm(Ks)
device = "cuda" if torch.cuda.is_available() else "cpu"
#if torch.backends.mps.is_available():
#    device = "mps"
for K in pbar:
    torch.manual_seed(1101252)
    N_samples = max(2*K, 128)
    z_0 = torch.randn(N_samples, N)
    z_block = torch.randn(N_samples, M, N)
    eps = torch.randn(N_samples,K, N)
    raw_patterns = sqrt(rho0)*z_0 + sqrt(rho1-rho0)* torch.repeat_interleave(z_block, repeats=B, dim=0) + sqrt(1-rho1)*eps
    uniform_weights = torch.ones(K)/K
    weights_fp = []
    weights_fp_centered = []
    for patterns_idx, patterns in enumerate(patterns_collection):
        pbar.set_description(f"K={K}, sample={patterns_idx+1}/{N_samples}")
        dyn_patterns =  DualDeterministicDynamics(patterns.to(device), biases=None, requires_grad=False).to(device)
        w_fp = dyn_patterns.integrate(uniform_weights[None,:].to(device), betas.to(device), num_iterations=3000).to('cpu')
        dyn_patterns_centered = DualDeterministicDynamics((patterns - patterns.mean(dim=0)).to(device), biases=None, requires_grad=False).to(device)
        w_fp_centered = dyn_patterns_centered.integrate(uniform_weights[None,:].to(device), betas.to(device), num_iterations=3000).to('cpu')
        weights_fp.append(w_fp)
        weights_fp_centered.append(w_fp_centered)
    weights_fp = torch.cat(weights_fp, dim=0)
    weights_fp_centered = torch.cat(weights_fp_centered, dim=0)
    H_fp = get_entropies(weights_fp).mean(dim=0)
    H_fp_centered = get_entropies(weights_fp_centered).mean(dim=0)
    torch.save({ "N" : N, "K": K, 
                "betas": betas.cpu().numpy(),
                 "patterns": patterns_collection.cpu().numpy(), 
                 "weights_fp": weights_fp.cpu().numpy(), 
                 "weights_fp_centered": weights_fp_centered.cpu().numpy()}, 
                 "local_data/random_patterns_N={}_K={}.pt".format(N, K))
    #fig, ax = plt.subplots()
    #ax.plot(betas.cpu(), H_fp.cpu(), label="FP")
    #ax.plot(betas.cpu(), H_fp_centered.cpu(), label="FP centered")
    #ax.set_xscale("log")
    #ax.set_xlabel("beta")
    #ax.set_ylabel("entropy")
    #ax.set_title(f"K={K}")
    #ax.legend()
    #plt.show()